# INTELIGENCIA ARTIFICAL PARA PREVER SE UM USUÁRIO IRA CANCELAR A ASSINATURA DO SPOTIFY
## Usando algoritmo Random Forest

### Tipo de aprendizado
####    -  Supervisionado de classificação
### Tipo de conhecimento
####    - Heuristíco

--------------------------------------------------------

# Funções de Normalização
### Gênero

In [1]:
def norm_gender(df_gender):
    for i, cell_gender in enumerate(df_gender["gender"]):
        if cell_gender == "Male":
            df_gender.iloc[i, df_gender.columns.get_loc("gender")] = 0
        elif cell_gender == "Female":
            df_gender.iloc[i, df_gender.columns.get_loc("gender")] = 1
        elif cell_gender == "Other":   
            df_gender.iloc[i, df_gender.columns.get_loc("gender")] = 2

### País

In [2]:
def norm_country(df_country):
    le = LabelEncoder()
    le.fit(df_country["country"])
    country_encoded = le.transform(df_country["country"])
    for i, cell_country in enumerate(df_country["country"]):
        df_country.iloc[i, df_country.columns.get_loc("country")] = country_encoded[i]    

### Tipo de assinatura

In [3]:
def norm_sub(df_sub):
    for i, cell_subscription in enumerate(df_sub["subscription_type"]):
        if cell_subscription == "Free":
            df_sub.iloc[i, df_sub.columns.get_loc("subscription_type")] = 0  
        elif cell_subscription == "Student":
            df_sub.iloc[i, df_sub.columns.get_loc("subscription_type")] = 1
        elif cell_subscription == "Family":
            df_sub.iloc[i, df_sub.columns.get_loc("subscription_type")] = 2
        elif cell_subscription == "Premium":
            df_sub.iloc[i, df_sub.columns.get_loc("subscription_type")] = 3

### TIpo de dispositivo

In [4]:
def norm_device_type(df_device_type): 
    for i, cell_device in enumerate(df_device_type["device_type"]):
        if cell_device == "Mobile":
            df_device_type.iloc[i, df_device_type.columns.get_loc("device_type")] = 0  
        elif cell_device == "Web":
            df_device_type.iloc[i, df_device_type.columns.get_loc("device_type")] = 1
        elif cell_device == "Desktop":     
            df_device_type.iloc[i, df_device_type.columns.get_loc("device_type")] = 2  

# IMPORTANDO BIBLIOTECAS

In [5]:
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTENC
import seaborn as sns


# Separando dados de treino e dados de teste e eliminando colunas desnecessárias

In [24]:
data_set = pd.read_csv("spotify_dataset.csv")
y = data_set["is_churned"]
X = data_set.drop(columns=["ads_listened_per_week","offline_listening","user_id"], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, stratify=y)

print(X_train)

      gender  age country subscription_type  listening_time  \
3637   Other   39      UK              Free             215   
7742   Other   16      PK           Premium             138   
1614  Female   56      PK           Premium             202   
7784   Other   26      CA           Student             253   
334   Female   38      CA              Free              89   
...      ...  ...     ...               ...             ...   
4719  Female   22      FR           Student             280   
5615  Female   30      US            Family             185   
1358  Female   17      PK           Premium              92   
1145   Other   28      UK           Premium             298   
7022   Other   54      PK            Family             268   

      songs_played_per_day  skip_rate device_type  is_churned  
3637                    11       0.52      Mobile           1  
7742                    99       0.46      Mobile           0  
1614                    82       0.41     Desktop  

### Fazendo oversampling para balancear o DataSet

In [26]:
smote = SMOTENC(categorical_features=[0,2,3,7],random_state=0)
X_train,  y_train = smote.fit_resample(X_train, y_train)

print(np.unique(np.array(y_train), return_counts=True))

(array([0, 1]), array([2964, 2964]))


### Normalizando dados de treino


In [27]:
norm_country(X_train)
norm_device_type(X_train)
norm_gender(X_train)
norm_sub(X_train)

### Normalizando dados de teste

In [28]:
norm_country(X_test)
norm_device_type(X_test)
norm_gender(X_test)
norm_sub(X_test)

### Rodando validação cruzada
#### Busca exaustiva de melhor hiper parâmetros

In [29]:

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 15, 20, 30],
    'min_samples_split': [2, 5, 10],
}

clf = GridSearchCV(
    RandomForestClassifier(random_state=0, class_weight='balanced'),
    param_grid,
    cv=5,
    scoring='f1'
)

clf.fit(X_train, y_train)

,estimator,RandomForestC...andom_state=0)
,param_grid,"{'max_depth': [None, 10, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [100, 200]}"
,scoring,'f1'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


### Usando Random Forest

### Fazendo a predição de uma amostra

In [30]:
clf.predict(X_test.head(1))

array([0])

In [31]:
print(y_test.head(1))

2166    0
Name: is_churned, dtype: int64


### Fazendo a predição de 4000 amostras

In [32]:
y_pred = clf.predict(X_test)

In [33]:
print(classification_report(np.array(y_test), y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2965
           1       1.00      1.00      1.00      1035

    accuracy                           1.00      4000
   macro avg       1.00      1.00      1.00      4000
weighted avg       1.00      1.00      1.00      4000



In [34]:
print(confusion_matrix(y_test, y_pred))

[[2965    0]
 [   0 1035]]
